# Dynamic Bayesian Network Basics: Construction and Inspection

A dynamic Bayesian network (DBN) extends a Bayesian network with
time-indexed variables. This notebook covers how to define dynamic states,
write template CPDs using conin's time-expression system, and assemble a
DBN.

See also the [constraints](constraints.ipynb) and
[inference](inference.ipynb) notebooks for how to add constraints and run
MAP queries on DBNs.

In [ ]:
from conin.dynamic_bayesian_network import DynamicDiscreteBayesianNetwork
from conin.bayesian_network import DiscreteCPD

## Defining Dynamic States

A DBN has **dynamic nodes** — variables that are replicated at each time
step. Their state spaces are defined via `dynamic_states`.

In [ ]:
dbn = DynamicDiscreteBayesianNetwork()

dbn.dynamic_states = {
    "W": ["Sunny", "Cloudy", "Rainy"],
    "T": ["Hot", "Mild", "Cold"],
}

print("Dynamic nodes: ", dbn.dynamic_nodes)
print("States of W:   ", dbn.dynamic_states["W"])

## Template CPDs and the Time Expression System

CPDs in a DBN reference time-indexed nodes using tuples like `("W", 0)` for
a specific time step, or `("W", dbn.t)` for a **template** that applies at
every time step. The expression `dbn.t - 1` refers to the previous time
step, enabling transition CPDs.

There are two kinds of CPDs:

1. **Initial CPDs** — defined at time 0, e.g., `node=("W", 0)`
2. **Transition CPDs** — defined as templates using `dbn.t`, e.g.,
   `node=("W", dbn.t)` with parents `[("W", dbn.t - 1)]`

At inference time, conin unrolls these templates into a static Bayesian
network covering the requested time range.

### Initial CPDs

In [ ]:
cpd_w_start = DiscreteCPD(
    node=("W", 0),
    values={"Sunny": 0.6, "Cloudy": 0.3, "Rainy": 0.1},
)

cpd_t_start = DiscreteCPD(
    node=("T", 0),
    values={"Hot": 0.5, "Mild": 0.4, "Cold": 0.1},
)

### Transition CPDs

A transition CPD uses `dbn.t` to define a template. The parent
`("W", dbn.t - 1)` refers to the weather at the previous time step.

In [ ]:
# Weather depends on previous weather
cpd_w_trans = DiscreteCPD(
    node=("W", dbn.t),
    parents=[("W", dbn.t - 1)],
    values={
        "Sunny": {"Sunny": 0.7, "Cloudy": 0.2, "Rainy": 0.1},
        "Cloudy": {"Sunny": 0.3, "Cloudy": 0.4, "Rainy": 0.3},
        "Rainy":  {"Sunny": 0.2, "Cloudy": 0.3, "Rainy": 0.5},
    },
)

# Temperature depends on previous temperature AND previous weather
cpd_t_trans = DiscreteCPD(
    node=("T", dbn.t),
    parents=[("T", dbn.t - 1), ("W", dbn.t - 1)],
    values={
        ("Hot",  "Sunny"):  {"Hot": 0.8, "Mild": 0.2, "Cold": 0.0},
        ("Hot",  "Cloudy"): {"Hot": 0.6, "Mild": 0.3, "Cold": 0.1},
        ("Hot",  "Rainy"):  {"Hot": 0.1, "Mild": 0.7, "Cold": 0.2},
        ("Mild", "Sunny"):  {"Hot": 0.7, "Mild": 0.2, "Cold": 0.1},
        ("Mild", "Cloudy"): {"Hot": 0.4, "Mild": 0.5, "Cold": 0.1},
        ("Mild", "Rainy"):  {"Hot": 0.2, "Mild": 0.3, "Cold": 0.5},
        ("Cold", "Sunny"):  {"Hot": 0.6, "Mild": 0.3, "Cold": 0.1},
        ("Cold", "Cloudy"): {"Hot": 0.3, "Mild": 0.4, "Cold": 0.3},
        ("Cold", "Rainy"):  {"Hot": 0.1, "Mild": 0.3, "Cold": 0.6},
    },
)

## Assembling the Network

In [ ]:
dbn.cpds = [cpd_w_start, cpd_w_trans, cpd_t_start, cpd_t_trans]

dbn.check_model()
print("Model is valid.")
print("Dynamic nodes:", dbn.dynamic_nodes)
print("# CPDs:       ", len(dbn.cpds))

## Inspecting CPDs

In [ ]:
for cpd in dbn.cpds:
    print(f"{cpd.node}  (parents={cpd.parents})")
    if isinstance(cpd.values, dict):
        for key, val in cpd.values.items():
            print(f"  {key}: {val}")
    else:
        print(f"  {cpd.values}")
    print()

## A Minimal Example

A single dynamic variable with a start CPD and a transition CPD.

In [ ]:
simple_dbn = DynamicDiscreteBayesianNetwork()
simple_dbn.dynamic_states = {"Z": [0, 1]}

simple_dbn.cpds = [
    DiscreteCPD(node=("Z", 0), values=[0.5, 0.5]),
    DiscreteCPD(
        node=("Z", simple_dbn.t),
        parents=[("Z", simple_dbn.t - 1)],
        values={0: [0.7, 0.3], 1: [0.8, 0.2]},
    ),
]
simple_dbn.check_model()

print("Dynamic nodes: ", simple_dbn.dynamic_nodes)
print("Dynamic states:", simple_dbn.dynamic_states)